In [2]:
# ==========================================
# 🏆 THE FINAL CHANCE: 100% Data + Local Validator
# ==========================================
!pip install python-crfsuite pandas scikit-learn -q

import os
import glob
import random
import tarfile
import pycrfsuite
import pandas as pd
from sklearn.metrics import classification_report

# 1. จัดการไฟล์ Corpus
corpus_tar = 'Corpus-LST20.tar.gz'
corpus_dir = 'LST20_Corpus_Extracted'

if not os.path.exists(corpus_tar):
    for root, dirs, files in os.walk('.'):
        if 'Corpus-LST20.tar.gz' in files:
            corpus_tar = os.path.join(root, 'Corpus-LST20.tar.gz')
            break

if os.path.exists(corpus_tar) and not os.path.exists(corpus_dir):
    print(f"📦 กำลังแตกไฟล์ {corpus_tar}...")
    with tarfile.open(corpus_tar, 'r:gz') as tar:
        tar.extractall(path=corpus_dir)

# 2. ค้นหาไฟล์ทั้งหมดและแบ่ง Train / Validation (ทดสอบมือ)
search_path = os.path.join(corpus_dir, '**', 'train', '**', '*.txt')
all_files = glob.glob(search_path, recursive=True)
if not all_files:
    all_files = glob.glob(os.path.join(corpus_dir, '**', '*.txt'), recursive=True)

# สุ่มไฟล์ 5% ไว้ทดสอบระบบมือ (ประมาณ 300-400 ไฟล์)
random.seed(42) # ล็อค Seed ให้ผลทดสอบเสถียร
random.shuffle(all_files)
split_idx = int(len(all_files) * 0.95)
train_files = all_files[:split_idx]
val_files = all_files[split_idx:]

print(f"📂 ไฟล์ทั้งหมด: {len(all_files)} | เทรนจริง: {len(train_files)} | ทดสอบมือ: {len(val_files)}")

# 3. ฟังก์ชันสกัด Features (จัดเต็ม Trigram เพราะ RAM ไม่เต็มแล้ว)
def get_char_type(c):
    if c.isspace(): return 'WS'
    if c.encode('utf-8').isalpha(): return 'EN'
    if c.isdigit(): return 'DIGIT'
    if '\u0e00' <= c <= '\u0e7f':
        if c in 'ะาำเแโใไๆฯ': return 'VOWEL'
        if c in '่้๊๋': return 'TONE'
        if c in '็์': return 'SIGN'
        if c in '0123456789': return 'TH_DIGIT'
        return 'TH_CONS'
    return 'OTHER'

def extract_features(chars):
    features = []
    for i in range(len(chars)):
        char = chars[i]
        # สังเกตว่าใช้ List ของ String (เป็นฟอร์แมตที่ประหยัด RAM ที่สุด)
        feat = ['bias', f'char={char}', f'char_type={get_char_type(char)}']

        # Window = 3
        for j in range(1, 4):
            if i - j >= 0:
                feat.append(f'-{j}:char={chars[i-j]}')
                feat.append(f'-{j}:char_type={get_char_type(chars[i-j])}')
            else:
                feat.append(f'-{j}:char=BOS')

            if i + j < len(chars):
                feat.append(f'+{j}:char={chars[i+j]}')
                feat.append(f'+{j}:char_type={get_char_type(chars[i+j])}')
            else:
                feat.append(f'+{j}:char=EOS')

        # Bigram & Trigram (ไม้ตายทะลุ 0.97)
        if i > 0: feat.append(f'-1:0:char={chars[i-1]+chars[i]}')
        if i < len(chars) - 1: feat.append(f'0:+1:char={chars[i]+chars[i+1]}')
        if i > 1: feat.append(f'-2:-1:0:char={chars[i-2]+chars[i-1]+chars[i]}')
        if i < len(chars) - 2: feat.append(f'0:+1:+2:char={chars[i]+chars[i+1]+chars[i+2]}')

        features.append(feat)
    return features

def generate_char_tags(words):
    chars, tags = [], []
    for word in words:
        if word == '_': word = ' '
        chars.extend(list(word))
        if len(word) == 1: tags.append('B_WORD')
        elif len(word) == 2: tags.extend(['B_WORD', 'E_WORD'])
        else:
            tags.append('B_WORD')
            tags.extend(['I_WORD'] * (len(word) - 2))
            tags.append('E_WORD')
    return chars, tags

# 4. เทรนโมเดลแบบ Iterative (ใช้ข้อมูล 100% โดย RAM ไม่พัง)
print("🧠 กำลังเทรนโมเดล (อ่านไฟล์ 100% อาจใช้เวลา 10-15 นาที ไปชงกาแฟรอได้เลย)...")
trainer = pycrfsuite.Trainer(verbose=False)

for filepath in train_files:
    with open(filepath, 'r', encoding='utf-8') as f:
        current_sent = []
        for line in f:
            line = line.strip()
            if not line:
                if current_sent:
                    chars, tags = generate_char_tags(current_sent)
                    trainer.append(extract_features(chars), tags)
                    current_sent = []
            else:
                current_sent.append(line.split('\t')[0])

trainer.set_params({
    'c1': 0.1,
    'c2': 0.1,
    'max_iterations': 120,
    'feature.possible_transitions': True
})
trainer.train('lst20_word_seg.crfsuite')
print("✅ เทรนเสร็จสมบูรณ์!")

# 5. ทอสอบมือ (Local Validation)
print("\n📊 กำลังจำลองการสอบ (Local Validation)...")
tagger = pycrfsuite.Tagger()
tagger.open('lst20_word_seg.crfsuite')

y_true_val, y_pred_val = [], []
for filepath in val_files:
    with open(filepath, 'r', encoding='utf-8') as f:
        current_sent = []
        for line in f:
            line = line.strip()
            if not line:
                if current_sent:
                    chars, tags = generate_char_tags(current_sent)
                    y_true_val.extend(tags)
                    y_pred_val.extend(tagger.tag(extract_features(chars)))
                    current_sent = []
            else:
                current_sent.append(line.split('\t')[0])

# ตรวจข้อสอบจำลอง
print("\n=== 🎯 ผลประเมินโมเดล (ถ้า F1 ทะลุ 0.975 แปลว่าพร้อมส่ง!) ===")
print(classification_report(y_true_val, y_pred_val, digits=4))

# 6. สร้างไฟล์ส่ง Kaggle
def find_file(filename):
    for root, dirs, files in os.walk('.'):
        if filename in files:
            return os.path.join(root, filename)
    return filename

print("\n💾 กำลังสร้างไฟล์ submission.csv สำหรับส่งจริง...")
ws_test_path = find_file('ws_test.txt')
ws_sample_path = find_file('ws_sample_submission.csv')

with open(ws_test_path, 'r', encoding='utf-8') as f:
    test_text = f.read()

X_test = extract_features(list(test_text))
y_pred_final = tagger.tag(X_test)

sample_df = pd.read_csv(ws_sample_path)
valid_ids = sample_df['Id'].tolist()

submission_data = []
allowed_tags = {'B_WORD', 'I_WORD', 'E_WORD'}

for char_idx in valid_ids:
    idx = char_idx - 1
    tag = y_pred_final[idx] if idx < len(y_pred_final) else 'I_WORD'
    if tag not in allowed_tags: tag = 'I_WORD'
    submission_data.append({'Id': char_idx, 'Predicted': tag})

sub_df = pd.DataFrame(submission_data)
sub_df.to_csv('submission.csv', index=False)

print(f"🎉 สร้างไฟล์สำเร็จแล้ว! แถวครบ {len(sub_df)} แถว โหลดไปกดส่งครั้งสุดท้ายได้เลยครับ!")

📂 ไฟล์ทั้งหมด: 3794 | เทรนจริง: 3604 | ทดสอบมือ: 190
🧠 กำลังเทรนโมเดล (อ่านไฟล์ 100% อาจใช้เวลา 10-15 นาที ไปชงกาแฟรอได้เลย)...
✅ เทรนเสร็จสมบูรณ์!

📊 กำลังจำลองการสอบ (Local Validation)...

=== 🎯 ผลประเมินโมเดล (ถ้า F1 ทะลุ 0.975 แปลว่าพร้อมส่ง!) ===
              precision    recall  f1-score   support

      B_WORD     0.9816    0.9846    0.9831    126143
      E_WORD     0.9776    0.9814    0.9795    104070
      I_WORD     0.9848    0.9816    0.9832    248670

    accuracy                         0.9824    478883
   macro avg     0.9813    0.9826    0.9819    478883
weighted avg     0.9824    0.9824    0.9824    478883


💾 กำลังสร้างไฟล์ submission.csv สำหรับส่งจริง...
🎉 สร้างไฟล์สำเร็จแล้ว! แถวครบ 35182 แถว โหลดไปกดส่งครั้งสุดท้ายได้เลยครับ!


In [3]:
import pandas as pd
import re

# 1. โหลดไฟล์ (แก้ชื่อไฟล์ให้ตรงกับที่อัปโหลด)
sub_file = 'submission.csv' # ชื่อไฟล์ที่คุณกำลังจะส่ง
test_file = 'ws_test.txt'       # ไฟล์ข้อสอบ

print("🔍 กำลังตรวจสอบไฟล์...")
sub_df = pd.read_csv(sub_file)
with open(test_file, 'r', encoding='utf-8') as f:
    text = f.read()

# 2. ตรวจสอบ Format พื้นฐาน
print("\n=== 1. ตรวจสอบความถูกต้องของ Format ===")
print(f"✅ จำนวนแถวที่ส่ง: {len(sub_df)} (มาตรฐานต้องเป็น 35,182 แถว)")
invalid_tags = sub_df[~sub_df['Predicted'].isin(['B_WORD', 'I_WORD', 'E_WORD'])]
if len(invalid_tags) > 0:
    print(f"❌ พบ Tag ประหลาดที่ไม่ใช่ B, I, E จำนวน {len(invalid_tags)} จุด!")
else:
    print("✅ Tag ทั้งหมดอยู่ในรูปแบบ B_WORD, I_WORD, E_WORD ถูกต้อง")

# 3. แปลงกลับเป็นข้อความเพื่อตรวจสอบด้วยตา (Visual Check)
print("\n=== 2. พรีวิวผลการตัดคำ (ประกอบร่างกลับเป็น Text) ===")
# สร้าง Dictionary Map Id -> Tag
tag_map = dict(zip(sub_df['Id'], sub_df['Predicted']))

segmented_text = ""
for i, char in enumerate(text, 1): # Id ตามโจทย์เริ่มที่ 1
    tag = tag_map.get(i, 'O') # ถ้าไม่มี Id นี้แปลว่าเป็นช่องว่าง

    if tag == 'B_WORD':
        segmented_text += "|" + char
    elif tag == 'I_WORD':
        segmented_text += char
    elif tag == 'E_WORD':
        segmented_text += char + "|"
    else: # ช่องว่าง (O)
        segmented_text += char

# ทำความสะอาดเครื่องหมาย | ที่อาจซ้อนกัน
segmented_text = re.sub(r'\|+', '|', segmented_text)
segmented_text = segmented_text.strip('|')

print("\n--- ลองอ่าน 1,000 ตัวอักษรแรกดูว่าสมเหตุสมผลไหม? ---")
print(segmented_text[:1000])

print("\n---------------------------------------------------")
print("💡 วิธีประเมินก่อนกดส่งจริง:")
print("1. ถ้าคุณเห็นคำถูกตัดสวยงาม เช่น '|ที่|ยัง|สถานการณ์|ยัง|ไม่|คลี่คลาย|อาจ|ส่ง|ผล|กระทบ|...' -> มั่นใจได้เลย กดส่งโลด!")
print("2. ถ้าคุณเห็นคำขาดวิ่นแปลกๆ เช่น '|สถานก|ารณ์|' หรือ '|กัมพูช|า|' -> หยุดก่อน! โมเดลมีปัญหา ต้องรันเทรนใหม่")

🔍 กำลังตรวจสอบไฟล์...

=== 1. ตรวจสอบความถูกต้องของ Format ===
✅ จำนวนแถวที่ส่ง: 35182 (มาตรฐานต้องเป็น 35,182 แถว)
✅ Tag ทั้งหมดอยู่ในรูปแบบ B_WORD, I_WORD, E_WORD ถูกต้อง

=== 2. พรีวิวผลการตัดคำ (ประกอบร่างกลับเป็น Text) ===

--- ลองอ่าน 1,000 ตัวอักษรแรกดูว่าสมเหตุสมผลไหม? ---
ที่|ยัง|สถานการณ์|ยัง|ไม่|คลี่คลาย|อาจ|ส่ง|ผล|กระทบการ|ค้า|ชายแดน|ไทย| |- |กัมพูชา| |ว่า| |เท่า|ที่|เข้า|ไป|ดำเนินการ|ตรวจสอบ|ยัง|ไม่|พบ|ว่า|มีการ|ปิด|ด่าน|บริเวณ|ดังกล่าว| |และ|ที่|สำคัญ|การ|ค้า|ระหว่าง|ไทย|และ|กัมพูชา|ส่วนใหญ่|จะ|มี|ปริมาณ|มาก|ที่|ด่าน|ปอยเปต| |อ.| |อรัญประเทศ| |จ.| |สระแก้ว| |มาก|กว่า|ด่าน|อื่น| |ๆ |ส่วน|ที่|กันทรลักษ์|เป็น|ด่าน|ขนาด|เล็ก|จะ|เน้น|ด้าน|ท่องเที่ยว| |จึง|ไม่|น่า|ส่ง|ผล|กระทบ|ต่อ|บรรยากาศ|การ|ค้า|ชายแดน|ไทย| |- |กัมพูชา|แต่อย่างใด| |อย่างไรก็ตาม| |แม้|จะ|มี|ปัญหา|ความ|ขัดแย้ง|เรื่อง|เขา|พระวิหาร| |แต่|ปริมาณ|การ|ค้า|ชายแดน|ไทย| |- |กัมพูชา|ใน|ปี|ที่|ผ่าน|มา|ยังคง|เพิ่ม|ขึ้น| |โดย|มี|จำ|ปริมาณ|ทั้งสิ้น| |48,406| |ล้าน|บาท| |เพิ่ม|ขึ้น|เมื่อ|เทียบ|กับ|ปี| |2549| |ร้อย|ละ| |0.2| 